# 09 — EDA và gợi ý cách chia persona

Mục tiêu: kiểm tra khác biệt **bên trong** nhóm event và so sánh rule với KMeans, cây quyết định. Không mặc định cluster đẹp là persona hữu ích, hoặc acquisition cao là lợi nhuận cao.

**Đầu vào:** output 08 `personas_YYYY-MM.parquet` và output 05 `lifecycle_YYYY-MM.parquet`. Không cần raw CSV. Thêm hai output vào Kaggle, sửa đường dẫn nếu tự tìm thấy nhiều phiên bản. Cần pandas/numpy/pyarrow/matplotlib/scikit-learn.

**Protocol:** fit tại 2015-11. Cây dùng nhãn Nov→Dec, có sẵn tại cutoff 2015-12; đánh giá source Dec2015–Apr2016. KMeans/scaler/biên M và số sản phẩm cố định từ tháng fit. Không refit trên evaluation. Các tháng đã xem trước đây nên đây là **development comparison**, không gọi là holdout mới. Chưa có kết quả Santander cho các phương án mới trong file này.

### Các hướng thử
1. **Event × danh mục:** kiểm tra 0,1,2,3,4,5,6+ sản phẩm trước khi quyết định biên; không tự động coi mọi danh mục hẹp tương đương.
2. **Event × M-score:** quantile M cố định tại tháng fit, riêng HasEvent/NoEvent. M là propensity upstream, không phải monetary. Phải kiểm tra nguồn M đã biết tại cutoff; tắt USE_M nếu chưa xác nhận được.
3. **KMeans hai nhánh:** tách NoEvent/HasEvent trước. NoEvent dùng độ rộng danh mục và lịch sử quan sát; HasEvent thêm R/F và tỷ lệ acquisition theo exposure. Không gán R giả cho NoEvent.
4. **Cây quyết định nông:** học các ngưỡng giải thích được từ acquisition tháng sau. Đây là phân nhóm có giám sát; không diễn giải thành động lực tâm lý hay nhân quả.

Hồ sơ tuổi/thu nhập và cấu trúc sản phẩm là hai hướng bổ sung sau: chưa có các cột đó trong output 08 nên file này không tự suy diễn.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import silhouette_score, average_precision_score
PERSONA_ROOT=None  # folder chứa personas_YYYY-MM.parquet của 08
LIFECYCLE_ROOT=None  # folder chứa lifecycle_YYYY-MM.parquet của 05
FIT_MONTH='2015-11'
EVAL_MONTHS=['2015-12','2016-01','2016-02','2016-03','2016-04']
USE_M=False  # chỉ True khi xác nhận M upstream không dùng thông tin tương lai
K_OPTIONS=[3,5]
FIT_SAMPLE=100000
TREE_SAMPLE=200000
SEED=42
MIN_N=1000;MIN_POS=20
OUT=Path('/kaggle/working/persona_eda_outputs' if Path('/kaggle/working').exists() else './persona_eda_outputs')
OUT.mkdir(parents=True,exist_ok=True)
ROOT=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
def resolve(configured,pattern):
    root=Path(configured) if configured else ROOT
    parents=sorted({p.parent for p in root.rglob(pattern)})
    if len(parents)!=1:raise ValueError(f'{pattern}: đặt đường dẫn duy nhất, tìm thấy {parents}')
    return parents[0]
def shift(m,j):return str(pd.Period(m,'M')+j)
P=resolve(PERSONA_ROOT,'personas_????-??.parquet')
L=resolve(LIFECYCLE_ROOT,'lifecycle_????-??.parquet')
assert FIT_MONTH<min(EVAL_MONTHS) and shift(FIT_MONTH,1)<=min(EVAL_MONTHS)
def load(m):
    d=pd.read_parquet(P/f'personas_{m}.parquet')
    needed=['customer_id','is_observed','products_complete','valid_pairs','R','F_event_months','n_products','event_history_group','persona']
    assert set(needed)<=set(d),'Cần output 08 bản event windows'
    assert d.customer_id.notna().all() and d.customer_id.is_unique
    d=d.loc[d.is_observed.eq(True)&d.products_complete.eq(True)].copy()
    d['branch']=np.where(d.F_event_months.eq(0),'NoEvent','HasEvent')
    d['exposure_rate']=d.F_event_months/d.valid_pairs.replace(0,np.nan)
    # Common cohort: đủ history, F/R hợp lệ; báo coverage riêng ngoài cohort.
    valid=d.valid_pairs.ge(3)&d.F_event_months.ge(0)&d.F_event_months.mod(1).eq(0)
    valid &= d.branch.eq('NoEvent') | (d.R.notna()&d.R.ge(0))
    valid &= d.n_products.between(0,24)&np.isfinite(d.exposure_rate)
    if USE_M:valid &= d.M_group_points.notna()&np.isfinite(d.M_group_points)&d.M_group_points.ge(0)
    return d.loc[valid].reset_index(drop=True),dict(month=m,current_complete_n=len(d),cohort_n=int(valid.sum()),excluded_n=int((~valid).sum()))
def target(d,m):
    z=pd.read_parquet(L/f'lifecycle_{shift(m,1)}.parquet',columns=['customer_id','acquisition_pair_valid','has_acquisition_month'])
    assert z.customer_id.is_unique
    assert not (z.acquisition_pair_valid.eq(True)&z.has_acquisition_month.isna()).any()
    j=d[['customer_id']].merge(z,on='customer_id',how='left',validate='one_to_one')
    valid=j.acquisition_pair_valid.eq(True).fillna(False).to_numpy(bool)
    y=j.has_acquisition_month.eq(True).fillna(False).to_numpy(int)
    return valid,y
fit,fit_coverage=load(FIT_MONTH)
fit_valid,fit_y=target(fit,FIT_MONTH)
print('Fit cohort',len(fit),'label coverage',fit_valid.mean(),'USE_M',USE_M)


## 1. EDA có điều kiện: nhóm nào đang bị gom quá rộng?
Mỗi bảng trả lời một câu hỏi: cùng event history nhưng khác danh mục có rate khác không? Cùng R nhưng khác F4 có khác không? Lịch sử quan sát ngắn có làm F thấp giả tạo không? Báo cả numerator/denominator và CI, không chỉ lift. Khi so HasEvent/NoEvent cần nhớ left truncation và exposure khác nhau.

Biên M học riêng từng branch ở tháng fit; không dùng quantile của các tháng evaluation. Bảng R×F chỉ xét HasEvent; R của NoEvent để thiếu.

In [ ]:
def bin_count(d):
    return pd.cut(d.n_products,[-1,0,1,2,3,4,5,np.inf],labels=['0','1','2','3','4','5','6+']).astype(str)
def f4(d):
    return pd.cut(d.F_event_months,[-1,0,1,2,4,np.inf],labels=['NoEvent','F1','F2','F3','F4']).astype(str)
M_EDGES={}
if USE_M:
    for branch,g in fit.groupby('branch'):
        e=np.unique(g.M_group_points.quantile([0,.25,.5,.75,1]).to_numpy())
        M_EDGES[branch]=np.r_[-np.inf,e[1:-1],np.inf]
def m_bucket(d):
    result=pd.Series('NotUsed',index=d.index)
    if USE_M:
        for b in M_EDGES:
            mask=d.branch.eq(b)
            result.loc[mask]=pd.cut(d.loc[mask,'M_group_points'],M_EDGES[b],labels=False).map(lambda x:f'M{int(x)+1}' if pd.notna(x) else 'Missing')
    return result

def wilson(k,n):
    n=np.asarray(n,float);k=np.asarray(k,float);ok=n>0
    lo=np.full_like(n,np.nan);hi=lo.copy();p=k[ok]/n[ok];nn=n[ok];z=1.96
    c=(p+z*z/(2*nn))/(1+z*z/nn);h=z*np.sqrt(p*(1-p)/nn+z*z/(4*nn*nn))/(1+z*z/nn)
    lo[ok]=np.maximum(0,c-h);hi[ok]=np.minimum(1,c+h);return lo,hi
eda=[];coverage=[]
for m in EVAL_MONTHS:
    d,a=load(m);ok,y=target(d,m);a.update(label_n=int(ok.sum()),label_coverage=float(ok.mean()));coverage.append(a)
    d['count_bin']=bin_count(d);d['F4']=f4(d);d['M_bucket']=m_bucket(d)
    d['R_bin']=pd.cut(d.R,[-1,0,2,5,np.inf],labels=['R0','R1_2','R3_5','R6plus']).astype(str)
    d['history_bin']=pd.cut(d.valid_pairs,[2,5,9,np.inf],labels=['3_5pairs','6_9pairs','10plus']).astype(str)
    d['eligible']=ok;d['positive']=ok&(y==1)
    views=[('event_x_count',['event_history_group','count_bin']),('R_x_F4',['R_bin','F4']),('event_x_exposure',['event_history_group','history_bin'])]
    if USE_M:views.append(('event_x_M',['event_history_group','M_bucket']))
    for name,cols in views:
        q=d.loc[d.branch.eq('HasEvent')] if name=='R_x_F4' else d
        a=q.groupby(cols,observed=True).agg(n=('customer_id','size'),label_n=('eligible','sum'),positive_n=('positive','sum')).reset_index()
        a['view']=name;a['month']=m
        a['group']=a[cols].astype(str).agg(' | '.join,axis=1)
        a['rate']=a.positive_n/a.label_n.replace(0,np.nan)
        a['coverage']=a.label_n/a.n;a['lift']=a.rate/(y[ok].mean() if ok.any() else np.nan)
        a['ci_low'],a['ci_high']=wilson(a.positive_n,a.label_n)
        a['weak_evidence']=(a.label_n<MIN_N)|(a.positive_n<MIN_POS)
        eda.append(a)
eda=pd.concat(eda,ignore_index=True);eda.to_csv(OUT/'conditional_eda.csv',index=False)
pd.DataFrame(coverage).to_csv(OUT/'cohort_coverage.csv',index=False)
# Điểm minh họa tháng cuối; toàn bộ 5 tháng trong CSV.
for name,g in eda.loc[eda.month.eq(EVAL_MONTHS[-1])].groupby('view'):
    print(name);display(g[['group','n','label_n','positive_n','rate','lift','ci_low','ci_high','weak_evidence']].sort_values('n',ascending=False).head(25))
q=eda.loc[eda.view.eq('event_x_count')&~eda.weak_evidence]
stability=q.groupby('group').agg(months=('month','nunique'),min_rate=('rate','min'),max_rate=('rate','max'),min_lift=('lift','min'),max_lift=('lift','max')).reset_index()
stability.to_csv(OUT/'conditional_rate_stability.csv',index=False);display(stability.sort_values('min_lift',ascending=False).head(20))


## 2. Phương án tự tìm nhóm: KMeans riêng HasEvent / NoEvent
Fit StandardScaler sau log1p các biến count, R/F; exposure_rate giữ nguyên. NoEvent không dùng R/F hằng số. Hai lựa chọn k=3,5 mỗi branch → tối đa 6/10 nhóm tổng. Đặc trưng exposure có thể làm nhóm phản ánh độ dài quan sát: kiểm tra profile trước khi đặt tên.

Nếu USE_M=True, KMeans thêm M; đó là cấu hình khác, không tự nhận là baseline đã độc lập với propensity score. Silhouette chỉ tính trong từng branch trên cùng sample, không dùng để xếp cây hoặc nhóm event có missing R.

In [ ]:
BRANCH_FEATURES={'NoEvent':['n_products','valid_pairs'],
'HasEvent':['R','F_event_months','n_products','valid_pairs','exposure_rate']}
if USE_M:
    for b in BRANCH_FEATURES:BRANCH_FEATURES[b].append('M_group_points')
def raw(d,cols):
    x=d[cols].to_numpy(float).copy()
    assert np.isfinite(x).all() and (x>=0).all()
    for j,c in enumerate(cols):
        if c!='exposure_rate':x[:,j]=np.log1p(x[:,j])
    return x
models={};geometry=[]
for k in K_OPTIONS:
    for b,cols in BRANCH_FEATURES.items():
        g=fit.loc[fit.branch.eq(b)]
        sample=g.sample(min(FIT_SAMPLE,len(g)),random_state=SEED)
        scaler=StandardScaler().fit(raw(sample,cols));x=scaler.transform(raw(sample,cols))
        if len(sample)<k or len(np.unique(x,axis=0))<k:
            print('Skip branch lacking distinct points',k,b);continue
        model=KMeans(n_clusters=k,n_init=10,random_state=SEED).fit(x)
        models[k,b]=(scaler,model,cols)
        ix=np.random.default_rng(SEED).choice(len(x),min(1500,len(x)),replace=False)
        labs=model.predict(x[ix]);unique=len(np.unique(labs))
        score=silhouette_score(x[ix],labs) if 1<unique<len(ix) else np.nan
        geometry.append(dict(k=k,branch=b,fit_sample_n=len(x),silhouette_sample_n=len(ix),silhouette=score,actual_groups=unique))
geometry=pd.DataFrame(geometry);geometry.to_csv(OUT/'kmeans_geometry.csv',index=False);display(geometry)
def km_labels(d,k):
    labels=pd.Series('MissingModel',index=d.index)
    for b in BRANCH_FEATURES:
        mask=d.branch.eq(b)
        if not mask.any() or (k,b) not in models:continue
        scaler,model,cols=models[k,b]
        labels.loc[mask]=[b+'_K'+str(i) for i in model.predict(scaler.transform(raw(d.loc[mask],cols)))]
    return labels


## 3. Cây nông học ngưỡng: persona theo khả năng acquisition
Fit nhãn tháng kế tiếp, tối đa 6 lá, mỗi lá ≥1% fit sample và ít nhất 100 khách. Giữ tỷ lệ lớp tự nhiên, không class_weight balanced để tránh dùng xác suất bị đổi prior. `has_event` phân biệt NoEvent với R điền 0; R=0 ở đây chỉ là mã hóa kỹ thuật của cây, không sửa R gốc. Cây có thể không tách tất cả 4 mức F: cần đọc rule đã học thay vì gọi đây là F4.

Chỉ dùng F/R, portfolio breadth, exposure và event flag; thêm M khi đã xác nhận as-of. Các lá là response segment, chưa phải chân dung marketing đầy đủ.

In [ ]:
TREE_COLS=['has_event','R_tree','F_event_months','n_products','valid_pairs','exposure_rate']
if USE_M:TREE_COLS.append('M_group_points')
def tree_frame(d):
    q=d.copy();q['has_event']=q.branch.eq('HasEvent').astype(int)
    q['R_tree']=q.R.where(q.has_event.eq(1),0)
    assert np.isfinite(q[TREE_COLS].to_numpy(float)).all()
    return q[TREE_COLS]
train=fit.loc[fit_valid].copy();train['y']=fit_y[fit_valid]
train=train.sample(min(TREE_SAMPLE,len(train)),random_state=SEED)
assert train.y.nunique()==2,'Cần đủ hai lớp tại tháng fit'
tree=DecisionTreeClassifier(max_leaf_nodes=6,min_samples_leaf=max(100,int(np.ceil(.01*len(train)))),random_state=SEED)
tree.fit(tree_frame(train),train.y)
rules=export_text(tree,feature_names=TREE_COLS,show_weights=True)
print(rules);(OUT/'tree_rules.txt').write_text(rules,encoding='utf-8')


## 4. So sánh trên cùng cohort, cùng nhãn, score khóa từ tháng fit
Mọi phương án được gán probability score từ rate nhóm Nov→Dec với smoothing prior_n=100. Cây cũng dùng rate trên cùng tập fit đầy đủ để so giao thức thống nhất. Không dùng rate evaluation tháng hiện tại để dự báo chính tháng đó. Nhóm fit chưa có dùng global fit baseline, báo fallback coverage.

Bảng gồm AP, Lift10, Brier, group counts, nhóm nhỏ, coverage; Lift10 dùng kỳ vọng ngẫu nhiên trong tie. Không chọn winner tự động trên development. Transition có thể phản ánh window aging; ARI đo mức giữ nhóm giữa hai tháng có cùng khách, không phải độ đúng.

In [ ]:
from sklearn.metrics import adjusted_rand_score
METHODS=['EventHistory','CurrentPersona','Tree6']+[f'KMeansBranches{k}' for k in K_OPTIONS if all((k,b) in models for b in BRANCH_FEATURES)]
if USE_M:METHODS+=['Event_MQuartile']
def assignments(d,method):
    if method=='EventHistory':return d.event_history_group.astype(str)
    if method=='CurrentPersona':return d.persona.astype(str)
    if method=='Tree6':return pd.Series(['Leaf'+str(i) for i in tree.apply(tree_frame(d))],index=d.index)
    if method=='Event_MQuartile':return d.event_history_group.astype(str)+'_'+m_bucket(d)
    return km_labels(d,int(method.removeprefix('KMeansBranches')))
prior=float(fit_y[fit_valid].mean());RATE_PRIOR_N=100
rate_tables={}
for method in METHODS:
    a=pd.DataFrame({'group':assignments(fit,method).to_numpy(),'y':fit_y,'valid':fit_valid})
    a=a.loc[a.valid].groupby('group').y.agg(['size','sum'])
    rate_tables[method]=(a['sum']+RATE_PRIOR_N*prior)/(a['size']+RATE_PRIOR_N)
def metrics(y,s):
    n=len(y);base=y.mean();k=max(1,int(.1*n));threshold=np.partition(s,n-k)[n-k]
    above=s>threshold;tie=s==threshold
    precision=(y[above].sum()+(k-above.sum())*y[tie].mean())/k
    return dict(baseline=base,AP=average_precision_score(y,s) if y.sum() else np.nan,
        Lift10=precision/base if base else np.nan,Brier=np.mean((s-y)**2))
results=[];profiles=[];trans=[];previous={}
for m in EVAL_MONTHS:
    d,a=load(m);ok,y=target(d,m)
    if not ok.any():continue
    saved=d[['customer_id']].copy()
    for method in METHODS:
        labels=assignments(d,method);scores=labels.map(rate_tables[method]);fallback=scores.isna()
        scores=scores.fillna(prior).to_numpy(float)
        counts=labels.value_counts()
        results.append(dict(month=m,method=method,n=len(d),label_n=int(ok.sum()),label_coverage=ok.mean(),groups=len(counts),
            small_group_share=counts[counts<MIN_N].sum()/len(d),fallback_share=fallback.mean(),**metrics(y[ok],scores[ok])))
        q=d.copy();q['group']=labels.to_numpy();q['label_valid']=ok;q['positive']=ok&(y==1)
        for label,g in q.groupby('group'):
            profiles.append(dict(month=m,method=method,group=label,n=len(g),label_n=int(g.label_valid.sum()),positive_n=int(g.positive.sum()),
                R_median=g.R.dropna().median() if g.R.notna().any() else np.nan,
                F_median=g.F_event_months.median(),products_median=g.n_products.median(),history_median=g.valid_pairs.median(),
                noevent_share=g.branch.eq('NoEvent').mean()))
        current=pd.DataFrame({'customer_id':d.customer_id,'group':labels})
        if method in previous:
            prev_m,prev=previous[method]
            j=prev.merge(current,on='customer_id',how='inner',validate='one_to_one',suffixes=('_from','_to'))
            if len(j):trans.append(dict(source_month=prev_m,target_month=m,method=method,common_n=len(j),source_n=len(prev),common_coverage=len(j)/len(prev),same_group_rate=j.group_from.eq(j.group_to).mean(),ARI=adjusted_rand_score(j.group_from,j.group_to)))
        previous[method]=(m,current)
        saved[method]=labels.to_numpy()
    saved.to_parquet(OUT/f'candidate_assignments_{m}.parquet',index=False)
results=pd.DataFrame(results);profiles=pd.DataFrame(profiles)
profiles['rate']=profiles.positive_n/profiles.label_n.replace(0,np.nan)
profiles['ci_low'],profiles['ci_high']=wilson(profiles.positive_n,profiles.label_n)
profiles['weak_evidence']=(profiles.label_n<MIN_N)|(profiles.positive_n<MIN_POS)
results.to_csv(OUT/'method_comparison_monthly.csv',index=False)
profiles.to_csv(OUT/'candidate_group_profiles.csv',index=False)
pd.DataFrame(trans).to_csv(OUT/'candidate_stability.csv',index=False)
summary=results.groupby('method').agg(months=('month','nunique'),AP_mean=('AP','mean'),Lift10_mean=('Lift10','mean'),Lift10_min=('Lift10','min'),Brier_mean=('Brier','mean'),groups_max=('groups','max'),small_share_max=('small_group_share','max'),fallback_max=('fallback_share','max'))
display(summary.round(4));summary.to_csv(OUT/'method_summary.csv')
display(pd.DataFrame(trans).round(4))
print('Không chọn winner tự động; đọc profile + độ tin cậy + độ ổn định trước khi đặt tên.')
(OUT/'protocol.json').write_text(json.dumps(dict(fit_month=FIT_MONTH,fit_label_month=shift(FIT_MONTH,1),eval_months=EVAL_MONTHS,USE_M=USE_M,branches=BRANCH_FEATURES,methods=METHODS,seed=SEED,scope='development backtest; shared eligible cohort; no forward refit'),ensure_ascii=False,indent=2),encoding='utf-8')


## 5. Chọn cách chia để báo cáo

| Quan sát sau EDA | Cách chia đáng thử |
|---|---|
| NoEvent có rate khác rõ theo 0/1/2/3/4+ sản phẩm | Persona theo độ rộng danh mục, biên chọn từ EDA rồi khóa và kiểm tra giai đoạn mới |
| M tạo khác biệt lớn trong cùng event group, nguồn M hợp lệ as-of | Event × propensity tier; ghi rõ đây là score tier, không phải giá trị tiền |
| KMeans có profile rõ, đủ lớn, rate khác và bền theo tháng | Đặt tên dựa trung vị/IQR; giữ NoEvent riêng; xem exposure có đang chi phối không |
| Cây nông cho rule dễ hiểu và Lift10 bền | Dùng leaf làm response persona, bổ sung lifecycle/portfolio để diễn giải |
| Tách thêm không cải thiện khác biệt hoặc chỉ tạo nhóm rất nhỏ | Giữ lớp tag chi tiết, không tăng số persona chính |

Không chốt chỉ bằng silhouette hoặc Lift10. Cần xem đồng thời: mô tả đặc điểm, coverage, min group size, CI theo tháng, tỷ lệ chuyển và khả năng hành động. Cây tối đa 6 lá chưa chắc tạo đủ 6 lá. KMeans hai nhánh tối đa 6/10 nhóm chưa chắc là số persona nên trình bày.

**Điền kết luận sau Run All:** nhóm nào hiện bị trộn? Đặc trưng nào giải thích khác biệt? Phương án nào giữ được lift thấp nhất qua các tháng? Có thay đổi do dài lịch sử thay vì hành vi không? Chọn 4–6 persona chính để trình bày nếu đủ bằng chứng; giữ các chiều khác dưới dạng tag.

Chưa thử cấu trúc sản phẩm: có thể nối output 06 với 24 biến sở hữu, gom product family hoặc phân cụm portfolio. Cần xác nhận tên/thứ tự bit portfolio_code từ manifest trước khi decode để đặt tên sản phẩm. Không giả định cùng số sản phẩm là cùng nhu cầu.